# Chapter 2. Model Predictive Control—Regulation
## 모델 예측 제어 — 조절

Rawlings, Mayne, Diehl (2017), 2판 1쇄, **Chapter 2, 인쇄 pp.89–192**를 바탕으로 한 한글 설명·실습 자료입니다.
교재의 2.1–2.12와 모든 하위 절 순서를 유지합니다. 페이지는 PDF 뷰어의 파일 페이지가 아닌 교재 인쇄 페이지입니다.

**학습 목표**
- 상태·입력·종단 제약이 있는 MPC를 정식화하고 실행 가능성을 확인한다.
- 종단 제어와 이전 해를 이동한 후보해로 재귀적 실행 가능성을 설명한다.
- 최적 값 함수의 감소를 수치적으로 검사하고 안정성 보장의 가정을 구분한다.
- 선형·주기적·비선형 모델과 차선 제어를 비교한다.
- 경제적 MPC의 평균 성능과 소산성을 구분하고, 이산 입력을 포함한 제어를 계산한다.

**선수 지식:** 1장의 이산시간 모델, LQR·리카티 방정식, 고유값, 이차계획법.

[PPT](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter02_mpc_regulation/slides/chapter02_slides.pptx)
· [PDF](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter02_mpc_regulation/slides/chapter02_slides.pdf)

**실행:** 코랩에서 `런타임 → 모두 실행`. CPU용입니다. 개별 실습은 공통 환경 준비 셀 이후 해당 실습 셀을 실행합니다.
그래프는 노트북과 현재 작업 폴더의 `figures/`에 저장됩니다. 설치와 한글 글꼴 다운로드에는 인터넷 연결이 필요합니다.

**범위:** 설명과 수식은 교재에 근거해 한국어로 재서술했습니다. Examples 2.5–2.6의 QP는 원본 수치와 해를 재현하고, Example 2.8은 실행 가능 입력의 구조를 계산합니다. 나머지는 **보충 실습**입니다.
교재의 모든 정리 증명과 연습문제 해답을 수록한 것은 아닙니다. 이 장의 기본 실습은 정확한 상태와 모델을 사용하는 **명목 MPC**입니다.

**목차:** 2.1 개요 / 2.2 MPC / 2.3 동적 계획법 / 2.4 안정성 / 2.5 MPC 사례 /
2.6 종단 제약의 필요성 / 2.7 차선 MPC / 2.8 경제적 MPC / 2.9 이산 구동기 / 2.10 정리 / 2.11 문헌 해설 / 2.12 연습문제

## 실습 환경 준비

교재 외 준비 항목입니다. 1장 실행 결과나 로컬 저장소가 필요하지 않습니다.
제어 비용은 `Q,R`, 종단 비용 행렬은 `P`, 최적 입력 부호는 `u=Kx`로 통일합니다.
QP의 최적화 상태와 수치 제약 위반량을 모두 확인합니다. 실패했을 때 영 입력을 반환해 성공처럼 진행하지 않습니다.

In [ ]:
import importlib.util, subprocess, sys

packages = {
    "numpy": "numpy>=2,<3",
    "scipy": "scipy>=1.14,<2",
    "matplotlib": "matplotlib>=3.9,<4",
    "cvxpy": "cvxpy>=1.6,<2",
    "osqp": "osqp>=1,<2",
}
missing = [
    spec for name, spec in packages.items() if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import scipy
from scipy.linalg import solve_discrete_are, expm
from scipy.signal import cont2discrete
import matplotlib
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path
import tempfile
import urllib.request
from matplotlib import font_manager

# Colab과 로컬에서 동일하게 한글 그래프를 표시하기 위한 공개 글꼴.
font_dir = Path(tempfile.gettempdir()) / "mpc_korean_fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic-Regular.ttf"
if not font_path.exists():
    url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    with urllib.request.urlopen(url, timeout=60) as response:
        font_path.write_bytes(response.read())
font_manager.fontManager.addfont(str(font_path))
korean_font = font_manager.FontProperties(fname=font_path).get_name()

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.8),
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 11,
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": ["DejaVu Sans", korean_font],
        "mathtext.fontset": "dejavusans",
        "axes.unicode_minus": False,
    }
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
CHECKS = {}


def finish_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()


def riccati(A, B, Q, R, Pf, N):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N은 양의 정수여야 합니다.")
    if not np.allclose(R, R.T) or np.linalg.eigvalsh(R).min() <= 0:
        raise ValueError("R은 대칭 양의 정부호여야 합니다.")
    # 부호 관례: u = K x. K에 음의 부호를 포함한다.
    Ps = [None] * (N + 1)
    Ks = [None] * N
    Ps[N] = Pf.copy()
    for k in range(N - 1, -1, -1):
        Pnext = Ps[k + 1]
        Ks[k] = -np.linalg.solve(R + B.T @ Pnext @ B, B.T @ Pnext @ A)
        Ps[k] = Q + A.T @ Pnext @ A + A.T @ Pnext @ B @ Ks[k]
        Ps[k] = (Ps[k] + Ps[k].T) / 2
    return Ps, Ks


def simulate_linear(A, B, policy, x0, steps):
    xs = [np.array(x0, dtype=float)]
    us = []
    for k in range(steps):
        u = np.atleast_1d(policy(xs[-1], k))
        us.append(u)
        xs.append(A @ xs[-1] + B @ u)
    return np.array(xs), np.array(us)


def checked_solve(problem):
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, max_iter=20000)
    if problem.status != cp.OPTIMAL or not np.isfinite(problem.value):
        raise RuntimeError(f"최적화 실패: {problem.status}")
    violation = max(
        (float(np.max(c.violation())) for c in problem.constraints), default=0.0
    )
    if violation > 2e-6:
        raise RuntimeError(f"제약 잔차가 큽니다: {violation:.2e}")
    return violation


In [ ]:
from itertools import product
from scipy.optimize import minimize
from scipy.linalg import solve_discrete_lyapunov


def scalar_parameters():
    a, b, q, r, umax, xmax, xf = 1.1, 1.0, 1.0, 0.2, 1.0, 4.0, 0.5
    P = float(solve_discrete_are([[a]], [[b]], [[q]], [[r]]).item())
    K = -b * P * a / (r + b * b * P)
    return a, b, q, r, P, K, umax, xmax, xf


def scalar_rollout(x0, u, a=1.1, b=1.0):
    x = np.empty(len(u) + 1)
    x[0] = float(x0)
    for k, uk in enumerate(u):
        x[k + 1] = a * x[k] + b * uk
    return x


def scalar_cost(x0, u, a=1.1, b=1.0, q=1.0, r=0.2, P=None):
    if P is None:
        P = scalar_parameters()[4]
    x = scalar_rollout(x0, u, a, b)
    return float(q * np.sum(x[:-1] ** 2) + r * np.sum(np.asarray(u) ** 2) + P * x[-1] ** 2)


def scalar_feasible(x0, u, a=1.1, b=1.0, umax=1.0, xmax=4.0, xf=0.5, tol=2e-6):
    x = scalar_rollout(x0, u, a, b)
    return (
        np.max(np.abs(u)) <= umax + tol
        and np.max(np.abs(x)) <= xmax + tol
        and (xf is None or abs(x[-1]) <= xf + tol)
    )


def make_scalar_mpc(N=6, a=1.1, b=1.0, q=1.0, r=0.2, P=None, umax=1.0, xmax=4.0, xf=0.5):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N은 양의 정수여야 합니다.")
    if not np.all(np.isfinite([a, b, q, r])) or q < 0 or r <= 0:
        raise ValueError("유한 모델 계수와 q≥0, r>0이 필요합니다.")
    if P is not None and (not np.isfinite(P) or P < 0):
        raise ValueError("종단 가중치는 유한한 비음수여야 합니다.")
    for bound in [umax, xmax, xf]:
        if bound is not None and (not np.isfinite(bound) or bound < 0):
            raise ValueError("제약 상한은 유한한 비음수여야 합니다.")
    if P is None:
        P = float(solve_discrete_are([[a]], [[b]], [[q]], [[r]]).item())
    x = cp.Variable(N + 1)
    u = cp.Variable(N)
    init = cp.Parameter()
    cons = [x[0] == init, x[1:] == a * x[:-1] + b * u]
    if umax is not None:
        cons += [cp.abs(u) <= umax]
    if xmax is not None:
        cons += [cp.abs(x) <= xmax]
    if xf is not None:
        cons += [cp.abs(x[-1]) <= xf]
    objective = q * cp.sum_squares(x[:-1]) + r * cp.sum_squares(u) + P * cp.square(x[-1])
    problem = cp.Problem(cp.Minimize(objective), cons)

    def solve(x0, allow_infeasible=False):
        if not np.isfinite(x0):
            raise ValueError("초기상태는 유한해야 합니다.")
        init.value = float(x0)
        problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, max_iter=30000)
        if allow_infeasible and problem.status == cp.INFEASIBLE:
            return None
        if problem.status != cp.OPTIMAL or not np.isfinite(problem.value):
            raise RuntimeError(problem.status)
        violation = max(float(np.max(c.violation())) for c in cons)
        if violation > 2e-6:
            raise RuntimeError(f"QP 제약 잔차가 큽니다: {violation:.2e}")
        return x.value.copy(), u.value.copy(), float(problem.value)

    return solve


print("제2장 공통 함수 준비 완료. 각 실습은 앞 실습의 결과에 의존하지 않습니다.")


def clip_polygon(vertices, normal, bound, tol=1e-12):
    """볼록 다각형을 반공간 normal @ x <= bound와 교차한다."""
    result = []
    for p, q in zip(vertices, np.roll(vertices, -1, axis=0)):
        dp, dq = normal @ p - bound, normal @ q - bound
        if dp <= tol:
            result.append(p)
        if (dp <= tol) != (dq <= tol):
            result.append(p + dp / (dp - dq) * (q - p))
    if not result:
        raise ValueError("교집합이 비었습니다.")
    return np.array(result)


## 2.1 Introduction — 개요

**교재 인쇄 p.89–91**

제약이 있는 제어에서는 현재 문제의 최적해만 구해서는 충분하지 않습니다. 적용 후의 상태에서도 다음 문제가 풀려야 하며, 반복되는 제어가 목표 상태로 수렴해야 합니다. 이 장은 실행 가능성과 안정성을 연결하는 설계 조건을 다룹니다.

기본 논의는 상태를 정확히 알고 모델과 실제 변화가 같은 명목 조건입니다. 외란에 대한 강인성은 3장, 출력 피드백은 5장에서 더 다룹니다.

## 2.2 Model Predictive Control — 모델 예측 제어

**교재 인쇄 p.91–107**

예측 길이 $N$에서 단계 비용과 종단 비용의 합을 최소화합니다. 모든 예측 단계의 상태·입력 쌍은 허용 집합에 있어야 하며 마지막 상태는 종단 집합에 있어야 합니다. 첫 입력만 적용한 뒤 새 상태에서 다시 풉니다.

실행 가능한 입력 열이 있는 초기상태의 집합이 $X_N$입니다. 연속성·닫힌 제약 집합·필요한 유계성은 해의 존재를 논하는 조건이며, 해의 유일성이나 안정성을 자동으로 뜻하지는 않습니다.

$$
\min_{u_0,\ldots,u_{N-1}}\ \sum_{j=0}^{N-1}\ell(x_j,u_j)+V_f(x_N)\quad\mathrm{s.t.}\quad x_0=x,\ x_{j+1}=f(x_j,u_j),\ (x_j,u_j)\in Z,\ x_N\in X_f
$$

**해의 존재·유일성·연속성은 서로 다른 질문입니다 (pp.96–106)**

연속인 비용을 비어 있지 않은 콤팩트 실행 가능 집합에서 최소화하면 최솟값이 존재합니다. 엄격한 볼록성은 볼록 집합 위에서 최적 입력의 유일성을 줄 수 있습니다. 그러나 모델과 비용이 연속이라고 제어법까지 자동으로 연속인 것은 아닙니다. 상태에 따라 바뀌는 실행 가능 집합의 성질도 필요합니다. 원본 정리 2.7과 Example 2.8을 구분해서 읽습니다.

**원본 Examples 2.5–2.6 재현 (pp.99–104):** $x^+=x+u$, $N=2$, $|u_j|\le1$, 상태·종단 제약 없음, $J=\frac12(x_0^2+x_1^2+x_2^2+u_0^2+u_1^2)$입니다. 상태를 제거하면
$$J=\tfrac32x^2+[2x\ \ x]U+\tfrac12U^THU,\qquad H=\begin{bmatrix}3&1\\1&2\end{bmatrix}.$$

$x\ge0$에서 최적 입력 열은 세 구간으로 나뉩니다:
- $0\le x\le5/3$: $U^*=[-3x/5,-x/5]$.
- $5/3\le x\le3$: $U^*=[-1,(1-x)/2]$.
- $x\ge3$: $U^*=[-1,-1]$. 음수 상태는 대칭을 이용합니다.

**읽는 법:** 타원 등고선이 입력 상자에 처음 닿는 점이 최적해입니다. 이 예제의 첫 입력은 무제약 유한 구간 LQ의 첫 입력을 자른 결과와 일치하지만, **전체 입력 열을 성분별로 잘라낸 해는 일반적으로 최적해가 아닙니다.** 원본 비용의 1/2는 이 실습에서 그대로 유지합니다.

**원본 수치 확인:** p.100은 초기상태 10의 최적 비용을 124로 표기합니다. 그러나 제시된 비용식과 상태 10,9,8·입력 -1,-1을 직접 대입하면 $(100+81+64+1+1)/2=123.5$입니다. 본 자료는 이 불일치를 명시하고 직접 계산값 123.5를 검증 기준으로 사용합니다.

In [ ]:
def demo_book_qp():
    H = np.array([[3.0, 1.0], [1.0, 2.0]])
    U = cp.Variable(2)
    init = cp.Parameter()
    objective = (
        1.5 * cp.square(init) + init * np.array([2.0, 1.0]) @ U + 0.5 * cp.quad_form(U, H)
    )
    problem = cp.Problem(cp.Minimize(objective), [cp.abs(U) <= 1])

    def solve(x):
        init.value = float(x)
        checked_solve(problem)
        return U.value.copy(), float(problem.value)

    def formula(x):
        z = abs(x)
        v = (
            np.array([-0.6 * z, -0.2 * z])
            if z <= 5 / 3
            else (np.array([-1.0, (1 - z) / 2]) if z <= 3 else -np.ones(2))
        )
        return np.sign(x) * v

    grid = np.linspace(-5, 5, 81)
    sol = []
    values = []
    errors = []
    for x in grid:
        u, val = solve(x)
        sol.append(u)
        values.append(val)
        errors.append(np.max(np.abs(u - formula(x))))
    u10, v10 = solve(10.0)
    assert np.max(errors) < 1e-6 and abs(v10 - 123.5) < 1e-6
    assert abs(v10 - 0.5 * (10**2 + 9**2 + 8**2 + 1 + 1)) < 1e-6
    # 독립적인 상태·입력 동시 최적화와 축약 표현을 대조한다.
    z = cp.Variable(3)
    v = cp.Variable(2)
    full = cp.Problem(
        cp.Minimize(0.5 * (cp.sum_squares(z) + cp.sum_squares(v))),
        [z[0] == 2.25, z[1:] == z[:-1] + v, cp.abs(v) <= 1],
    )
    checked_solve(full)
    reduced, value = solve(2.25)
    assert np.max(np.abs(v.value - reduced)) < 1e-7 and abs(full.value - value) < 1e-7
    center = -np.linalg.solve(H, np.array([2.0, 1.0]) * 2.25)
    clipped = np.clip(center, -1, 1)

    def J(v):
        return 1.5 * 2.25**2 + np.array([4.5, 2.25]) @ v + 0.5 * v @ H @ v

    gap = J(clipped) - J(reduced)
    assert gap > 0.02
    CHECKS["원본 2.5 x=10 최적 비용"] = v10
    CHECKS["원본 구간별 입력식 QP 차이"] = float(max(errors))
    CHECKS["단순 clipping 비용 손실"] = float(gap)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    sol = np.array(sol)
    ax[0].plot(grid, sol[:, 0], label="첫 최적 입력")
    ax[0].plot(grid, sol[:, 1], label="둘째 최적 입력")
    for cut in [-3, -5 / 3, 5 / 3, 3]:
        ax[0].axvline(cut, color="gray", ls=":", alpha=0.5)
    ax[0].set(xlabel="초기상태 x", ylabel="최적 입력")
    ax[0].legend()
    ax[1].plot(grid, values)
    ax[1].set(xlabel="초기상태 x", ylabel="최적 값 함수")
    finish_figure("book_piecewise_qp")
    xx, yy = np.meshgrid(np.linspace(-2, 1, 200), np.linspace(-1.5, 1, 200))
    pts = np.stack([xx - center[0], yy - center[1]], axis=-1)
    cost = 0.5 * np.einsum("...i,ij,...j->...", pts, H, pts)
    plt.figure(figsize=(8, 4))
    plt.contour(
        xx, yy, cost, levels=[0.03, 0.1, 0.3, 0.5, 1, 2, 3], colors="gray", alpha=0.6
    )
    plt.plot(
        [-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color="#168b86", label="허용 입력 상자"
    )
    for label, p, marker in [
        ("무제약 최적점", center, "x"),
        ("성분별 clipping", clipped, "s"),
        ("제약 최적점", reduced, "o"),
    ]:
        plt.scatter(*p, s=65, marker=marker, label=label)
    plt.xlabel("첫 입력 u₀")
    plt.ylabel("둘째 입력 u₁")
    plt.legend(fontsize=9)
    plt.gca().set_aspect("equal")
    finish_figure("book_qp_geometry")
    return {"grid": grid, "inputs": sol, "values": np.array(values), "clipping_gap": gap}


book_qp_result = demo_book_qp()


**보충 실습 A — 상태·입력·종단 제약을 모두 넣은 스칼라 MPC**

모델은 $x^+=1.1x+u$, 비용은 $\ell=x^2+0.2u^2$, 제약은 $|x|\le4$, $|u|\le1$, 종단 집합은 $|x_N|\le0.5$입니다.
종단 비용은 DARE 해로 구합니다. 각 시점에서 초기상태부터 마지막 상태까지 제약 위반량을 직접 계산합니다.

In [ ]:
def demo_regulation():
    a, b, q, r, P, K, umax, xmax, xf = scalar_parameters()
    solve = make_scalar_mpc()
    x = 3.0
    xs = [x]
    us = []
    predictions = []
    violation = 0.0
    for k in range(24):
        xp, up, value = solve(x)
        predictions.append(xp)
        violation = max(
            violation,
            np.max(np.abs(xp)) - xmax,
            np.max(np.abs(up)) - umax,
            abs(xp[-1]) - xf,
        )
        assert abs(xp[0] - x) < 1e-6
        assert np.max(np.abs(xp[1:] - a * xp[:-1] - b * up)) < 1e-6
        us.append(up[0])
        x = a * x + b * up[0]
        xs.append(x)
    assert violation < 2e-6 and abs(xs[-1]) < 1e-5
    CHECKS["상태·입력·종단 최대 위반량"] = float(violation)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for k in range(0, 7, 2):
        ax[0].plot(np.arange(k, k + 7), predictions[k], "--", alpha=0.35)
    ax[0].plot(xs, "o-", label="폐루프 상태")
    ax[0].axhspan(-xf, xf, alpha=0.13, color="#168b86", label="종단 집합")
    ax[0].axhline(xmax, color="gray", ls=":")
    ax[0].axhline(-xmax, color="gray", ls=":")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[1].step(range(len(us)), us, where="post", label="적용 입력")
    for v in [-umax, umax]:
        ax[1].axhline(v, color="gray", ls=":")
    ax[1].set(xlabel="시간 단계", ylabel="입력")
    for axy in ax:
        axy.legend()
    finish_figure("constrained_regulation")


demo_regulation()


**원본 Example 2.8의 실행 가능 구조 (pp.105–107)**

$x_1^+=x_1+u$, $x_2^+=x_2+u^3$, $N=3$, $x(3)=0$인 비선형 문제입니다. 첫 입력 $v=u_0$를 정하면 나머지는 $u_1+u_2=s=-x_1-v$, $u_1^3+u_2^3=t=-x_2-v^3$을 만족해야 합니다. $s\ne0$일 때 $u_1u_2=(s^3-t)/(3s)$이므로 이차방정식의 판별식이 비음수여야 실수 입력이 존재합니다.

그림은 원본처럼 초기상태를 단위원 위에서 움직이며 **첫 입력의 실행 가능 여부만** 계산합니다. 원본의 불연속 제어법·값 함수 그림 전체를 재현하거나 전역 최적해를 계산한 것은 아닙니다. 비용이 매끄러워도 실행 가능 입력 구간이 갈라질 수 있음을 보여 줍니다. 특이 경우 $s=0$에서는 $t=0$이면 $u_1=-u_2$가 모두 가능하고, 아니면 해가 없습니다.

In [ ]:
def demo_book_nonlinear_feasibility():
    theta = np.linspace(-np.pi, np.pi, 361)
    first = np.linspace(-2, 2, 401)
    tt, vv = np.meshgrid(theta, first)
    x1, x2 = np.cos(tt), np.sin(tt)
    s = -x1 - vv
    t = -x2 - vv**3
    regular = np.abs(s) > 1e-10
    product_pair = np.divide(s**3 - t, 3 * s, out=np.full_like(s, np.nan), where=regular)
    disc = s * s - 4 * product_pair
    feasible = (regular & (disc >= 0)) | ((~regular) & (np.abs(t) < 1e-10))
    # 표시 격자의 비특이 부분에서 두 후속 입력을 복원해 원래 종단 방정식 대조.
    test = regular & (disc >= 0) & (np.abs(s) > 0.05) & (disc < 100)
    u1 = (s[test] + np.sqrt(disc[test])) / 2
    u2 = (s[test] - np.sqrt(disc[test])) / 2
    residual = max(
        np.max(np.abs(x1[test] + vv[test] + u1 + u2)),
        np.max(np.abs(x2[test] + vv[test] ** 3 + u1**3 + u2**3)),
    )
    assert residual < 1e-9
    CHECKS["원본 2.8 복원 종단 잔차"] = float(residual)
    plt.figure(figsize=(10, 4))
    plt.pcolormesh(
        theta / np.pi, first, feasible, shading="auto", cmap="Greys", vmin=0, vmax=1
    )
    plt.xlabel("단위원 각도 θ / π")
    plt.ylabel("첫 입력 u₀")
    plt.title("검정: 종단 원점에 도달하는 후속 입력 존재 · 흰색: 불가능")
    finish_figure("book_nonlinear_feasibility")
    return {"theta": theta, "first_input": first, "feasible": feasible}


nonlinear_feasibility_result = demo_book_nonlinear_feasibility()


## 2.3 Dynamic Programming Solution — 동적 계획법 해법

**교재 인쇄 p.107–111**

남은 구간의 최적 비용을 값 함수로 요약하면 한 단계 최적화로 재귀식을 쓸 수 있습니다. 제약이 있는 경우에는 다음 상태가 나머지 문제의 실행 가능 집합에 속해야 합니다. 종단 집합에서 출발해 전임 집합을 반복 계산하면 $X_N$을 구성할 수 있습니다.

종단 집합이 허용 제어 아래 불변이면 기존 입력 열 뒤에 종단 제어를 붙일 수 있어 예측 길이에 따른 실행 가능 집합의 포함 관계가 성립합니다.

DP는 모든 초기상태에서의 값 함수와 정책을 구하는 관점입니다. 상태 격자는 차원이 늘수록 급격히 커지므로 일반적인 비선형·제약 문제에서 실용적이지 않을 수 있습니다. MPC는 매 시점 현재 상태에서 필요한 최적화만 풀어 이 정책을 구현합니다.

**보충 실습 B — 실행 가능 집합의 역방향 계산**

이 모델에서는 $X_N=[-r_N,r_N]$이고 $r_0=0.5$, $r_{N+1}=\min(4,(r_N+1)/1.1)$입니다.
구간 경계의 수치적 모호성을 피한 격자점에서 이 조건과 실제 QP 결과가 일치하는지 확인합니다.

In [ ]:
def demo_feasible_sets():
    radii = [0.5]
    for _ in range(7):
        radii.append(min(4.0, (radii[-1] + 1.0) / 1.1))
    assert np.all(np.diff(radii) >= -1e-12)
    checked = 0
    for N in [1, 2, 4]:
        solve = make_scalar_mpc(N=N)
        for x in np.linspace(-4, 4, 19):
            if abs(abs(x) - radii[N]) < 1e-4:
                continue
            feasible = solve(x, allow_infeasible=True) is not None
            assert feasible == (abs(x) < radii[N] + 1e-6)
            checked += 1
    CHECKS["실행 가능 집합 QP 대조점 수"] = checked
    plt.figure()
    for N, radius in enumerate(radii):
        plt.plot([-radius, radius], [N, N], lw=8, solid_capstyle="round", alpha=0.75)
    plt.xlabel("초기상태")
    plt.ylabel("예측 길이 N")
    plt.yticks(range(8))
    finish_figure("feasible_sets")


demo_feasible_sets()


## 2.4 Stability — 안정성

**교재 인쇄 p.112–129**

최적화 해가 존재하는 것과 반복 제어가 안정한 것은 별개의 문제입니다. 이 절에서는 값 함수가 상태 크기를 위아래에서 제한하고, 폐루프를 따라 감소하는지 확인합니다. 단일 궤적에서 값이 줄어든다는 관측은 전체 영역에 대한 증명의 대체물이 아닙니다.

### 2.4.1 Introduction — 안정성 분석의 출발점

**교재 인쇄 p.112–114**

안정성은 작은 초기 편차가 작게 유지되는 성질이고, 점근 안정성은 여기에 원점으로의 수렴을 더한 성질입니다. MPC의 값 함수는 제약 경계에서 미분 가능하지 않을 수 있으므로 도함수 대신 이산시간의 함수값 차이를 사용합니다.

### 2.4.2 Stabilizing Conditions — 안정화 조건

**교재 인쇄 p.114–120**

종단 집합 안에서 허용되는 종단 제어 $\kappa_f$를 선택하여 다음 상태도 종단 집합 안에 머물게 하고 종단 비용 감소가 단계 비용을 상쇄하도록 합니다. 여기에 비용의 양의 하한과 필요한 상한 조건을 결합합니다.

현재 최적 입력 열의 첫 항을 버리고 마지막에 종단 제어를 붙이면 다음 문제의 실행 가능한 후보해가 됩니다. 다음 최적해는 이 후보보다 비싸지 않으므로 값 함수 감소를 얻습니다. 이 논리는 모델과 실제 다음 상태가 일치한다는 조건을 사용합니다.

$$
\tilde{\mathbf u}=(u_1^*,\ldots,u_{N-1}^*,\kappa_f(x_N^*)),\qquad V_N^0(x^+)\le V_N(x^+,\tilde{\mathbf u})\le V_N^0(x)-\ell(x,u_0^*)
$$

In [ ]:
def demo_stability():
    a, b, q, r, P, K, umax, xmax, xf = scalar_parameters()
    assert abs(K) * xf <= umax and abs(a + b * K) * xf <= xf
    residual = (a + b * K) ** 2 * P - P + q + r * K * K
    assert abs(residual) < 1e-10
    solve = make_scalar_mpc()
    x = 3.0
    vals = []
    residuals = []
    candidate_gaps = []
    for k in range(25):
        xp, up, value = solve(x)
        xnext = a * x + b * up[0]
        shifted = np.r_[up[1:], K * xp[-1]]
        assert scalar_feasible(xnext, shifted)
        candidate = scalar_cost(xnext, shifted)
        _, _, vnext = solve(xnext)
        stage = q * x * x + r * up[0] ** 2
        assert vnext <= candidate + 2e-6
        assert candidate <= value - stage + 2e-6
        vals.append(value)
        residuals.append(vnext - value + stage)
        candidate_gaps.append(candidate - vnext)
        x = xnext
    CHECKS["최적 비용 감소 잔차 최대값"] = float(max(residuals))
    CHECKS["종단 리카티 등식 잔차"] = float(abs(residual))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].semilogy(np.maximum(vals, 1e-16), "o-")
    ax[0].set(xlabel="시간 단계", ylabel="최적 값 함수")
    ax[1].plot(residuals, "o-")
    ax[1].axhline(0, color="gray", ls=":")
    ax[1].set(xlabel="시간 단계", ylabel="감소 부등식 잔차")
    finish_figure("lyapunov_decrease")


demo_stability()


**보충 실습 — 이동 후보해의 시간축과 명목 조건 (pp.114–120)**

$N=2$의 실행 가능 경계에서 시작합니다. 첫 입력 뒤 나머지 입력을 이동하고 종단 피드백을 붙인 후보는 **명목 다음 상태**에서는 실행 가능합니다. 실제 다음 상태에만 충격 +0.4를 더하면 같은 입력 열의 종단 제약이 깨집니다. 그러나 재최적화는 여전히 실행 가능할 수 있습니다. 따라서 이 그림은 재귀적 실행 가능성의 증명에 쓰인 후보가 깨졌음을 보이며, 외란이 있으면 항상 MPC 문제 전체가 불가능하다는 뜻은 아닙니다.

In [ ]:
def demo_shifted_candidate():
    a, b, q, r, P, K, umax, xmax, xf = scalar_parameters()
    N = 2
    radius = ((xf + 1) / a + 1) / a
    solve = make_scalar_mpc(N=N)
    xp, up, value = solve(radius - 1e-6)
    shifted = np.r_[up[1:], K * xp[-1]]
    nominal = scalar_rollout(xp[1], shifted)
    actual = scalar_rollout(xp[1] + 0.4, shifted)
    assert scalar_feasible(xp[1], shifted)
    assert not scalar_feasible(xp[1] + 0.4, shifted)
    repaired, repair_u, _ = solve(xp[1] + 0.4)
    assert abs(repaired[-1]) <= xf + 2e-6
    CHECKS["외란 후 이동 후보 종단 위반"] = float(abs(actual[-1]) - xf)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(np.arange(3), xp, "o-", label="k=0의 기존 예측")
    ax[0].plot(np.arange(1, 4), nominal, "s--", label="k=1의 이동 후보")
    ax[0].set(xlabel="절대 시간 단계", ylabel="예측 상태")
    ax[0].legend()
    ax[1].plot(np.arange(1, 4), nominal, "o-", label="명목 이동 후보")
    ax[1].plot(np.arange(1, 4), actual, "o--", label="충격 +0.4, 같은 입력 열")
    ax[1].plot(np.arange(1, 4), repaired, "s-", label="충격 후 재최적화")
    for axis in ax:
        axis.axhspan(-xf, xf, color="#168b86", alpha=0.12)
    ax[1].set(xlabel="절대 시간 단계", ylabel="예측 상태")
    ax[1].legend(fontsize=9)
    finish_figure("shifted_candidate")
    return {"nominal": nominal, "disturbed": actual, "reoptimized": repaired}


shift_result = demo_shifted_candidate()


### 2.4.3 Exponential Stability — 지수 안정성

**교재 인쇄 p.120**

값 함수가 상태 노름의 제곱에 비례하는 상한·하한을 갖고 감소량에도 같은 차수의 하한이 있으면 기하급수적인 수렴률을 얻을 수 있습니다. 반로그 그래프의 직선처럼 보이는 모양만으로 지수 안정성을 증명할 수는 없습니다.

**보충 실습 — 그래프 모양 대신 수렴률의 상한 계산 (p.120)**

일반적으로 $c_1\|x\|^2\le V(x)\le c_2\|x\|^2$, $V(x^+)-V(x)\le-c_3\|x\|^2$이면 $V(x_k)\le(1-c_3/c_2)^kV(x_0)$입니다. 아래는 제약이 비활성인 작은 종단 집합 안에서 $V=Px^2$, $c_1=c_2=P$, $c_3=q$를 사용합니다. 이 경우에 대한 해석적 상한이며, 전체 제약 영역에 같은 상수를 무조건 적용하지 않습니다.

In [ ]:
def demo_exponential_bound():
    a, b, q, r, P, K, umax, xmax, xf = scalar_parameters()
    x0 = 0.4
    steps = 18
    factor = 1 - q / P
    assert 0 < factor < 1
    state = x0 * (a + b * K) ** np.arange(steps + 1)
    bound = abs(x0) * factor ** (np.arange(steps + 1) / 2)
    assert np.all(np.abs(state) <= bound + 1e-14)
    assert abs(K) * xf <= umax and abs(a + b * K) < 1
    CHECKS["지수 값 함수 수축 상한"] = float(factor)
    plt.figure()
    plt.semilogy(np.maximum(np.abs(state), 1e-16), "o-", label="실제 상태 크기")
    plt.semilogy(bound, "--", label="Lyapunov 감소에서 얻은 상한")
    plt.xlabel("시간 단계")
    plt.ylabel("상태 크기")
    plt.legend()
    finish_figure("exponential_bound")
    return {"state": state, "bound": bound}


exponential_result = demo_exponential_bound()


### 2.4.4 Controllability and Observability — 제어가능성과 관측가능성

**교재 인쇄 p.120–123**

상태 전체 대신 출력과 입력에만 비용을 부과하면 단계 비용이 상태에 대해 양의 정부호가 아닐 수 있습니다. 이때 제어가능성·검출가능성 등 조건으로 비용에 드러나지 않는 상태의 거동을 다뤄야 합니다. 출력 비용이 작다고 모든 상태가 작다는 결론을 바로 내릴 수는 없습니다.

**보충 실습 — 출력 비용은 줄지만 숨은 상태는 발산한다 (pp.120–123 연결)**

$A=\mathrm{diag}(1.2,0.7)$, $B=[0,1]^T$, $C=[0,1]$이고 비용은 $y^2+0.2u^2$입니다. 첫 상태는 측정 비용에 나타나지 않고 제어로도 바꿀 수 없습니다. 둘째 상태의 최적 조절과 무관하게 첫 상태는 $1.2^k$로 발산합니다. 이것은 정리의 반례가 아니라 **검출 가능성 가정을 없앴을 때 결론을 적용할 수 없는 사례**입니다.

In [ ]:
def demo_hidden_unstable_mode():
    A = np.diag([1.2, 0.7])
    B = np.array([[0.0], [1.0]])
    C = np.array([[0.0, 1.0]])
    P = float(solve_discrete_are([[0.7]], [[1.0]], [[1.0]], [[0.2]]).item())
    K2 = -P * 0.7 / (0.2 + P)
    state, inputs = simulate_linear(A, B, lambda x, k: [K2 * x[1]], [1.0, 1.0], 30)
    stage = state[:-1, 1] ** 2 + 0.2 * inputs[:, 0] ** 2
    assert state[-1, 0] > 200 and abs(state[-1, 1]) < 1e-10
    assert np.linalg.matrix_rank(np.vstack([C, C @ A])) == 1
    CHECKS["숨은 불안정 상태 최종 크기"] = float(state[-1, 0])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].semilogy(np.maximum(np.abs(state), 1e-16))
    ax[0].legend(["보이지 않는 첫 상태", "비용으로 조절하는 둘째 상태"])
    ax[0].set(xlabel="시간 단계", ylabel="상태 크기")
    ax[1].semilogy(np.maximum(stage, 1e-16))
    ax[1].set(xlabel="시간 단계", ylabel="출력·입력 단계 비용")
    finish_figure("hidden_unstable_mode")
    return {"states": state, "stage_cost": stage}


hidden_result = demo_hidden_unstable_mode()


### 2.4.5 Time-Varying Systems — 시변 시스템

**교재 인쇄 p.123–129**

모델·비용·제약이 시간에 의존하면 값 함수와 종단 조건에도 시간을 포함합니다. 다음 시점의 문제에서는 모델뿐 아니라 종단 비용과 종단 집합의 시점도 한 단계 이동해야 합니다. 균일한 안정성 결과에는 시간에 대해 공통으로 성립하는 경계 조건이 필요합니다.

## 2.5 Examples of MPC — MPC 사례

**교재 인쇄 p.130–143**

교재는 안정화 조건이 서로 다른 모델과 제약 구조에서 어떻게 충족되는지 설명합니다. 각 사례에서 종단 제어, 종단 비용, 종단 집합의 역할을 구분하는 것이 중요합니다. 아래 실습은 간단한 모델로 그 구조를 확인하는 보충 예제입니다.

### 2.5.1 The Unconstrained Linear Quadratic Regulator — 제약 없는 선형 이차 조절기

**교재 인쇄 p.132–133**

무제약 선형 이차 문제에서 안정화 DARE 해를 종단 비용으로 사용하면 모든 리카티 단계에서 같은 행렬을 얻습니다. 따라서 예측 길이가 달라도 첫 입력은 무한 구간 LQR과 같습니다. 이 결과는 제약이 활성화되는 경우까지 그대로 확장되지 않습니다.

In [ ]:
def demo_lqr_equivalence():
    a, b, q, r, P, K, *_ = scalar_parameters()
    errors = []
    for N in [1, 2, 6, 12]:
        solve = make_scalar_mpc(N=N, P=P, umax=None, xmax=None, xf=None)
        for x0 in [-3.0, -0.5, 1.0, 3.0]:
            xp, up, val = solve(x0)
            errors.append(abs(up[0] - K * x0))
            assert abs(val - P * x0 * x0) < 1e-5
    assert max(errors) < 1e-6
    CHECKS["무제약 MPC·LQR 입력 최대 차이"] = float(max(errors))
    print("MPC와 LQR의 최대 입력 차이:", max(errors))


demo_lqr_equivalence()


### 2.5.2 Unconstrained Linear Periodic Systems — 제약 없는 선형 주기 시스템

**교재 인쇄 p.133–134**

모델이 일정 주기로 반복되면 종단 비용과 피드백도 위상별로 반복될 수 있습니다. 주기 리카티 방정식을 풀고 현재 시점과 예측 종료 시점의 위상을 맞춰야 합니다. 한 주기 동안의 폐루프 전이행렬로 안정성을 확인할 수 있습니다.

In [ ]:
def demo_periodic():
    a = np.array([1.15, 0.85])
    b = 0.5
    q = 1.0
    r = 0.2
    P = np.ones(2)
    for iteration in range(10000):
        new = np.array(
            [
                q
                + a[j] ** 2 * P[(j + 1) % 2]
                - (a[j] * b * P[(j + 1) % 2]) ** 2 / (r + b * b * P[(j + 1) % 2])
                for j in range(2)
            ]
        )
        change = np.max(np.abs(new - P))
        P = new
        if change < 1e-12:
            break
    else:
        raise RuntimeError("주기 리카티 재귀 미수렴")
    K = np.array(
        [-b * P[(j + 1) % 2] * a[j] / (r + b * b * P[(j + 1) % 2]) for j in range(2)]
    )
    multiplier = float(np.prod(a + b * K))
    assert abs(multiplier) < 1
    xs = [2.0]
    us = []
    errors = []
    N = 5
    for k in range(24):
        x = cp.Variable(N + 1)
        u = cp.Variable(N)
        cons = [x[0] == xs[-1]]
        cost = P[(k + N) % 2] * cp.square(x[-1])
        for j in range(N):
            cost += q * cp.square(x[j]) + r * cp.square(u[j])
            cons += [x[j + 1] == a[(k + j) % 2] * x[j] + b * u[j]]
        problem = cp.Problem(cp.Minimize(cost), cons)
        checked_solve(problem)
        errors.append(abs(u.value[0] - K[k % 2] * xs[-1]))
        us.append(u.value[0])
        xs.append(a[k % 2] * xs[-1] + b * u.value[0])
    assert max(errors) < 1e-6
    CHECKS["주기 MPC·리카티 입력 차이"] = float(max(errors))
    CHECKS["주기 폐루프 승수"] = abs(multiplier)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(xs, "o-")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[1].step(range(24), a[np.arange(24) % 2], where="post", label="모델 계수")
    ax[1].set(xlabel="시간 단계", ylabel="계수 a(k)")
    ax[1].legend()
    finish_figure("periodic_mpc")
    horizons = np.arange(1, 7)
    correct = []
    wrong = []

    def first_gain(N, terminal):
        value = terminal
        for j in range(N - 1, -1, -1):
            gain = -b * value * a[j % 2] / (r + b * b * value)
            value = q + a[j % 2] ** 2 * value + a[j % 2] * value * b * gain
        return gain

    for horizon in horizons:
        correct.append(abs(first_gain(horizon, P[horizon % 2]) - K[0]))
        wrong.append(abs(first_gain(horizon, P[0]) - K[0]))
    assert max(correct) < 1e-10 and wrong[0] > 1e-3
    assert max(np.array(wrong)[horizons % 2 == 0]) < 1e-10
    CHECKS["잘못된 종단 위상 N1 이득 차이"] = float(wrong[0])
    plt.figure()
    plt.semilogy(horizons, np.maximum(correct, 1e-16), "o-", label="종료 시점 위상")
    plt.semilogy(horizons, np.maximum(wrong, 1e-16), "s--", label="현재 위상으로 고정")
    plt.xlabel("예측 길이 N")
    plt.ylabel("정확한 주기 이득과의 차이")
    plt.legend()
    finish_figure("periodic_phase")


demo_periodic()


**위상 비교 해석:** 현재 위상을 0으로 두었습니다. 주기 2에서 짝수 구간은 종료 위상이 같지만 홀수 구간은 다릅니다. 잘못된 종단 위상의 영향이 긴 구간에서 작아질 수 있어도 두 문제가 동일하다는 뜻은 아닙니다.

### 2.5.3 Stable Linear Systems with Control Constraints — 안정한 선형 시스템의 입력 제약

**교재 인쇄 p.134–136**

개방루프가 안정하고 0 입력이 허용되며 상태 제약이 없다면 종단 제어를 0으로 선택할 수 있습니다. 이때 이산 리아푸노프 방정식으로 미래의 상태 비용을 표현하는 종단 비용을 구합니다. 모든 상태가 종단 후보가 될 수 있지만, 상태 제약이 추가되면 별도 불변 집합 검토가 필요합니다.

In [ ]:
def demo_stable_input_only():
    a = 0.8
    q = 1.0
    P = float(solve_discrete_lyapunov([[a]], [[q]]).item())
    assert abs(a * a * P - P + q) < 1e-12
    solve = make_scalar_mpc(a=a, P=P, xmax=None, xf=None, umax=0.4)
    for initial in [-10.0, 0.0, 10.0]:
        x = initial
        for _ in range(50):
            xp, up, v = solve(x)
            assert np.max(np.abs(up)) <= 0.4 + 1e-6
            x = a * x + up[0]
        assert abs(x) < 1e-5
    CHECKS["안정 모델 종단 등식 잔차"] = abs(a * a * P - P + q)
    print("안정 모델의 입력 제약 실습 통과")


demo_stable_input_only()


### 2.5.4 Linear Systems with Control and State Constraints — 상태·입력 제약이 있는 선형 시스템

**교재 인쇄 p.136–139**

상태 제약까지 있으면 종단 제어가 입력 제약뿐 아니라 다음 상태 제약도 지키는 영역을 선택해야 합니다. LQR 종단 제어를 사용하는 경우 해당 피드백 아래 양의 불변 집합을 구성하고 MPC의 마지막 상태를 그 안으로 제한합니다.

실습 A와 안정성 실습은 $X_f=[-0.5,0.5]$에서 입력 한계와 수축 조건을 직접 검사했습니다. 이는 해당 스칼라 모델의 보충 예제이며 교재의 다변수 예제 재현은 아닙니다.

**보충 실습 — 종단 반경의 정확한 한계와 이동 후보 반례**

기본 스칼라 모델 $x^+=ax+bu$, $a=1.1,b=1$에서 DARE 종단 이득 $K$를 고정합니다. $X_f=[-\rho,\rho]$의 모든 점이 상태·입력 제약을 지키려면 $\rho\le x_{\max}$, $|K|\rho\le u_{\max}$여야 합니다. $|a+bK|<1$이므로 이 조건 아래 종단 집합은 불변입니다. 따라서 이 **고정 피드백에 대한** 최대 허용 반경은
$$\rho_{\max}=\min(x_{\max},u_{\max}/|K|).$$

DARE 항등식 $P(a+bK)^2-P+q+rK^2=0$은 모든 $x$에서 비용 감소를 보이지만 입력 한계를 보장하지 않습니다. $\rho=1.5\rho_{\max}$로 키우고 $N=1$, $x_0=(\rho+u_{\max})/a$를 선택하면 최적 입력 $u_0=-1$과 종단 상태 $x_1=\rho$는 현재 문제에 실행 가능합니다. 그러나 이동 후보 $Kx_1$은 입력 한계를 위반합니다.

아래 코드는 두 경계점에서 종단 제어의 제약을 검사하고 QP로 반례를 독립 확인합니다. 후보가 실패해도 다음 문제를 다시 풀 수 있으므로, 이것은 확대된 집합에 대한 **이 종단 피드백의 증명 조건 실패**입니다. MPC 전체의 불가능성이나 불안정성의 증명은 아닙니다. 임의 제어를 허용한 최대 제어 불변 집합과도 구분합니다.


In [ ]:
def demo_terminal_radius(enlargement=1.5):
    a, b, q, r, P, K, umax, xmax, _ = scalar_parameters()
    closed_loop = a + b * K
    assert abs(closed_loop) < 1 and abs(K) > 0
    radius_max = min(xmax, umax / abs(K))
    endpoints = np.array([-radius_max, radius_max])
    assert np.max(np.abs(K * endpoints)) <= umax + 1e-12
    assert np.max(np.abs(closed_loop * endpoints)) <= radius_max + 1e-12
    dare_residual = P * closed_loop ** 2 - P + q + r * K ** 2
    assert abs(dare_residual) < 1e-12

    # 비용 감소는 그대로 성립해도 확대된 집합의 종단 입력은 허용되지 않는다.
    if not np.isfinite(enlargement) or enlargement <= 1:
        raise ValueError("확대 배율은 1보다 큰 유한한 수여야 합니다.")
    enlarged = enlargement * radius_max
    x0 = (enlarged + b * umax) / a
    if enlarged >= xmax or x0 > xmax:
        raise ValueError("이 반례는 확대 반경과 초기상태가 상태 한계 안에 있어야 합니다.")
    solve = make_scalar_mpc(N=1, xf=enlarged)
    xp, up, _ = solve(x0)
    assert np.isclose(up[0], -umax, atol=2e-6)
    assert np.isclose(xp[-1], enlarged, atol=2e-6)
    assert scalar_feasible(x0, up, xf=enlarged)
    shifted = np.array([K * xp[-1]])
    violation = float(abs(shifted[0]) - umax)
    expected_violation = abs(K) * enlarged - umax
    assert expected_violation > 0
    assert np.isclose(violation, expected_violation, atol=2e-6, rtol=1e-7)
    assert not scalar_feasible(xp[1], shifted, xf=enlarged)
    repaired_x, repaired_u, _ = solve(xp[1])
    assert scalar_feasible(xp[1], repaired_u, xf=enlarged)
    assert abs(dare_residual * enlarged ** 2) < 1e-11
    CHECKS["종단 고정 K 최대 허용 반경"] = float(radius_max)
    CHECKS["확대 종단 집합 이동 입력 위반"] = violation
    CHECKS["종단 DARE 비용 항등식 잔차"] = float(abs(dare_residual))

    radii = np.linspace(0, enlarged * 1.1, 100)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(radii, abs(K) * radii, label="경계의 최대 종단 입력 |K|ρ")
    axes[0].axhline(umax, color="black", linestyle="--", label="입력 한계")
    axes[0].axvline(radius_max, color="#168b86", linestyle=":", label="최대 허용 반경")
    axes[0].set(xlabel="종단 반경 ρ", ylabel="입력 크기")
    axes[0].legend(fontsize=9)
    axes[1].bar(["현재 최적 입력", "다음 이동 후보", "다음 재최적화"],
                [up[0], shifted[0], repaired_u[0]], color=["#168b86", "#d96b43", "#168b86"])
    axes[1].axhline(-umax, color="black", linestyle="--", label="입력 하한")
    axes[1].set(ylabel="제어 입력")
    axes[1].legend(fontsize=9)
    finish_figure("terminal_radius_certificate")
    print(f"고정 K 최대 반경={radius_max:.6f}, 확대 반경={enlarged:.6f}, 이동 입력 위반={violation:.6f}")
    return {"radius_max": radius_max, "enlarged_radius": enlarged,
            "current_states": xp, "shifted_input": shifted, "reoptimized_states": repaired_x}


terminal_radius_result = demo_terminal_radius()


**보충 실습 — 2차원 종단 불변 집합을 실제로 구성 (pp.136–139)**

이중 적분기 $A=\begin{bmatrix}1&0.4\\0&1\end{bmatrix}$, $B=[0.08,0.4]^T$, $Q=\mathrm{diag}(1,0.2)$, $R=0.2$를 사용합니다. 상태 한계는 위치 ±3, 속도 ±2, 입력 한계 ±1입니다. DARE로 얻은 $K$에 대해 초기 허용 집합 $S_0=\{x:Fx\le g\}$에는 **상태 제약과 $|Kx|\le1$**을 함께 넣습니다.

$S_{j+1}=S_j\cap\{x:(A+BK)x\in S_j\}$로 제한을 추가합니다. 구현은 $F(A+BK)^ix\le g$를 순서대로 추가하며, 다음 제약이 다각형 모든 꼭짓점에서 만족되는지 확인해 중단합니다. 선형 함수의 다각형 최댓값은 꼭짓점에서 발생하므로 이 검사는 격자 샘플보다 강합니다. 아래는 이 모델과 고정된 K의 최대 허용 불변 집합이며 임의 제어를 허용한 최대 제어 불변 집합과는 다릅니다.

함께 그리는 타원 $x^TPx\le\alpha$는 $\alpha=\min_i g_i^2/(F_iP^{-1}F_i^T)$로 선택합니다. 타원은 더 보수적이지만 비용 감소로 불변성을 쉽게 보입니다. 다각형 종단 제약은 QP를 유지하지만 타원 제약은 일반적인 선형 제약 QP 범위를 벗어납니다.

In [ ]:
def demo_terminal_polytope():
    A = np.array([[1.0, 0.4], [0.0, 1.0]])
    B = np.array([[0.08], [0.4]])
    Q = np.diag([1.0, 0.2])
    R = np.array([[0.2]])
    P = solve_discrete_are(A, B, Q, R)
    K = -np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
    Ak = A + B @ K
    F = np.vstack([np.eye(2), -np.eye(2), K, -K])
    g = np.array([3.0, 2.0, 3.0, 2.0, 1.0, 1.0])
    poly = np.array([[-3.0, -2.0], [3.0, -2.0], [3.0, 2.0], [-3.0, 2.0]])
    for f, bound in zip(F, g):
        poly = clip_polygon(poly, f, bound)
    initial = poly.copy()
    H = F.copy()
    bounds = g.copy()
    power = np.eye(2)
    for iteration in range(1, 61):
        power = power @ Ak
        new = F @ power
        if np.max(new @ poly.T - g[:, None]) <= 1e-10:
            break
        for f, bound in zip(new, g):
            poly = clip_polygon(poly, f, bound)
        H = np.vstack([H, new])
        bounds = np.r_[bounds, g]
    else:
        raise RuntimeError("불변 집합 반복이 60단계에서 수렴하지 않았습니다.")
    residual = float(np.max(H @ Ak @ poly.T - bounds[:, None]))
    assert residual < 1e-9
    denominator = np.einsum("ij,ji->i", F, np.linalg.solve(P, F.T))
    alpha = float(np.min(g * g / denominator))
    eig, vec = np.linalg.eigh(P)
    theta = np.linspace(0, 2 * np.pi, 301)
    ellipse = (
        np.sqrt(alpha) * (vec / np.sqrt(eig)) @ np.vstack([np.cos(theta), np.sin(theta)])
    )
    assert np.max(F @ ellipse - g[:, None]) < 1e-9
    assert np.linalg.eigvalsh(Ak.T @ P @ Ak - P).max() < 0
    CHECKS["다각형 종단 불변성 최대 잔차"] = residual
    CHECKS["종단 타원 alpha"] = alpha
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].fill(initial[:, 0], initial[:, 1], alpha=0.18, label="현재 상태·입력만 허용")
    ax[0].fill(poly[:, 0], poly[:, 1], alpha=0.35, label="고정 LQR의 불변 다각형")
    ax[0].plot(ellipse[0], ellipse[1], color="black", label="보수적인 비용 타원")
    for x0 in poly[:: max(1, len(poly) // 5)]:
        xs, _ = simulate_linear(A, B, lambda x, k: K @ x, x0, 12)
        ax[1].plot(xs[:, 0], xs[:, 1], "o-", ms=3)
    closed = np.vstack([poly, poly[0]])
    ax[1].plot(closed[:, 0], closed[:, 1], color="black", ls="--", label="불변 다각형 경계")
    for axis in ax:
        axis.set(xlabel="위치", ylabel="속도")
        axis.legend(fontsize=8)
        axis.set_aspect("equal")
    finish_figure("terminal_polytope")
    return {
        "vertices": poly,
        "P": P,
        "K": K,
        "alpha": alpha,
        "invariance_residual": residual,
    }


terminal_polytope_result = demo_terminal_polytope()


### 2.5.5 Constrained Nonlinear Systems — 제약이 있는 비선형 시스템

**교재 인쇄 p.139–141**

비선형 시스템에서도 종단 불변성과 비용 감소라는 논리 구조는 유지됩니다. 하지만 최적화가 비볼록이면 수치 솔버는 국소해를 반환할 수 있습니다. 실행 가능한 해인지, 이전 후보보다 비용이 증가하지 않았는지, 어떤 보장 조건을 충족하는지 구분해서 확인해야 합니다.

### 2.5.6 Constrained Nonlinear Time-Varying Systems — 제약이 있는 비선형 시변 시스템

**교재 인쇄 p.141–143**

비선형 모델이 시간에 따라 달라지면 예측 구간 안의 각 단계에 해당 시점의 모델을 사용해야 합니다. 현재 시점의 계수를 구간 전체에 고정하는 것은 다른 최적화 문제입니다. 아래에서는 시불변 모델과 정확히 알려진 시변 모델을 각각 제어합니다.

**보충 실습 C — 비선형 직접 사격법과 시변 모델**

짧은 입력 열을 최적화하고 모델을 앞으로 적분하듯 반복 계산합니다. 실현 가능한 초기 입력 열을 구성한 뒤 SLSQP로 개선합니다.
솔버 성공뿐 아니라 종단 잔차·상태 및 입력 제약·초기 후보 대비 비용도 확인합니다. 첫 단계 이후에도 이를 반복합니다.

In [ ]:
def demo_nonlinear():
    N = 6
    all_x = []
    max_residual = 0.0
    for varying in [False, True]:
        xs = [0.8]
        us = []

        def step(x, u, k):
            a = 0.9 + 0.1 * np.sin(0.3 * k) if varying else 0.9
            return a * x + 0.08 * x**3 + 0.5 * u

        for k in range(22):

            def rollout(U):
                z = [xs[-1]]
                for j, uj in enumerate(U):
                    z.append(step(z[-1], uj, k + j))
                return np.array(z)

            def cost(U):
                z = rollout(U)
                return np.sum(z[:-1] ** 2) + 0.2 * np.sum(U**2)

            z = xs[-1]
            seed = []
            for j in range(N):
                uj = -1.2 * z if j < N - 1 else -2 * step(z, 0, k + j)
                seed.append(uj)
                z = step(z, uj, k + j)
            seed = np.array(seed)
            assert np.max(np.abs(seed)) <= 1 and abs(rollout(seed)[-1]) < 1e-10
            result = minimize(
                cost,
                seed,
                method="SLSQP",
                bounds=[(-1, 1)] * N,
                constraints=[
                    {"type": "eq", "fun": lambda U: rollout(U)[-1]},
                    {"type": "ineq", "fun": lambda U: 1.5 - np.abs(rollout(U))},
                ],
                options={"ftol": 1e-10, "maxiter": 200},
            )
            if not result.success:
                raise RuntimeError(result.message)
            xp = rollout(result.x)
            residual = max(
                abs(xp[-1]), np.max(np.abs(xp)) - 1.5, np.max(np.abs(result.x)) - 1
            )
            assert residual < 1e-6 and result.fun <= cost(seed) + 1e-8
            max_residual = max(max_residual, residual)
            us.append(result.x[0])
            xs.append(step(xs[-1], us[-1], k))
        assert abs(xs[-1]) < 1e-5
        all_x.append(("시변 모델" if varying else "시불변 모델", xs, us))
    CHECKS["비선형 MPC 종단·제약 잔차"] = float(max_residual)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for label, xs, us in all_x:
        ax[0].plot(xs, "o-", label=label)
        ax[1].step(range(len(us)), us, where="post", label=label)
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[1].set(xlabel="시간 단계", ylabel="입력")
    for axy in ax:
        axy.legend()
    finish_figure("nonlinear_mpc")


demo_nonlinear()


## 2.6 Is a Terminal Constraint Set Xf Necessary? — 종단 제약 집합은 반드시 필요한가?

**교재 인쇄 p.144–147**

종단 제약은 안정성을 보이는 유용한 설계 장치이지만 모든 문제에서 필수인 것은 아닙니다. 적절한 종단 비용이나 충분한 예측 길이 등 추가 조건으로 종단 제약 없이도 안정성을 얻을 수 있습니다. 반대로 제약을 없앴다는 이유만으로 안정성이 유지되지는 않습니다.

아래 실험은 제약이 없는 스칼라 LQ 문제입니다. 종단 비용이 0인 짧은 구간과 DARE 종단 비용을 비교해, 종단 비용과 종단 집합이 서로 다른 설계 요소임을 보여 줍니다.

In [ ]:
def demo_terminal_design():
    A = np.array([[1.2]])
    B = np.array([[1.0]])
    Q = np.array([[1.0]])
    R = np.array([[0.2]])
    P = solve_discrete_are(A, B, Q, R)
    Ns = np.arange(1, 11)
    rho_zero = []
    rho_dare = []
    for N in Ns:
        for Pf, target in [(np.zeros((1, 1)), rho_zero), (P, rho_dare)]:
            _, Ks = riccati(A, B, Q, R, Pf, int(N))
            target.append(abs(float((A + B @ Ks[0]).item())))
    assert abs(rho_zero[0] - 1.2) < 1e-12
    assert max(rho_dare) < 1 and np.ptp(rho_dare) < 1e-10
    CHECKS["종단 비용 없는 N1 폐루프 계수"] = rho_zero[0]
    plt.figure()
    plt.plot(Ns, rho_zero, "o-", label="종단 비용 0")
    plt.plot(Ns, rho_dare, "s-", label="DARE 종단 비용")
    plt.axhline(1, color="gray", ls="--", label="안정성 경계")
    plt.xlabel("예측 길이 N")
    plt.ylabel("폐루프 계수의 절댓값")
    plt.legend()
    finish_figure("terminal_design")


demo_terminal_design()


## 2.7 Suboptimal MPC — 차선 MPC

**교재 인쇄 p.147–150**

매 시점 전역 최적해를 구하지 않아도 안정적인 제어를 구성할 수 있습니다. 핵심은 실행 가능한 입력 열을 유지하고, 보장된 후보보다 비용이 커지지 않는 해를 선택하는 것입니다. 이전에 보관한 입력 열을 이동한 뒤 종단 제어를 추가하면 다음 후보를 얻습니다.

단순히 최적화 반복 횟수를 제한하는 것만으로는 이 조건을 보장하지 않습니다. 실행 가능성 및 비용 개선을 명시적으로 확인하거나 안전한 후보를 유지해야 합니다.

**보충 실습 D — 실행 가능한 좌표 탐색으로 차선해 유지**

초기 QP로 입력 열을 구하고, 다음 시점부터는 각 입력을 작은 폭으로 바꾸어 실행 가능하고 비용이 개선되는 경우만 채택합니다.
첫 QP 해와 별도의 실행 가능 열을 절반씩 섞어 의도적으로 차선 후보에서 시작합니다.
제약과 비용 조건을 통과하지 못하면 이동 후보를 유지합니다. 이는 속도 벤치마크가 아니라 차선 제어의 수락 조건을 보여 주는 실습입니다.

In [ ]:
def demo_suboptimal():
    a, b, q, r, P, K, *_ = scalar_parameters()
    solve = make_scalar_mpc()
    x = 3.0
    _, optimum, _ = solve(x)
    feasible_seed = np.array([-1.0, -1.0, -1.0, -0.5, 0.0, 0.0])
    assert scalar_feasible(x, feasible_seed)
    candidate = 0.5 * optimum + 0.5 * feasible_seed
    xs = [x]
    vals = []
    us = []
    accepted = 0
    for k in range(24):
        assert scalar_feasible(x, candidate)
        seed_cost = scalar_cost(x, candidate)
        U = candidate.copy()
        value = seed_cost
        for sweep in range(2):
            for j in range(len(U)):
                for delta in [-0.05, 0.05]:
                    trial = U.copy()
                    trial[j] += delta
                    if scalar_feasible(x, trial):
                        trial_cost = scalar_cost(x, trial)
                        if trial_cost < value - 1e-12:
                            U = trial
                            value = trial_cost
                            accepted += 1
        assert value <= seed_cost + 1e-9
        if vals:
            assert value <= vals[-1] - (q * xs[-2] ** 2 + r * us[-1] ** 2) + 2e-6
        vals.append(value)
        us.append(U[0])
        xp = scalar_rollout(x, U)
        x = xp[1]
        candidate = np.r_[U[1:], K * xp[-1]]
        xs.append(x)
    assert abs(xs[-1]) < 0.01 and accepted > 0
    CHECKS["차선 MPC 마지막 상태"] = abs(float(xs[-1]))
    CHECKS["차선 탐색 개선 수락 횟수"] = accepted
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(xs, "o-")
    ax[0].set(xlabel="시간 단계", ylabel="차선 제어 상태")
    ax[1].semilogy(np.maximum(vals, 1e-16), "o-")
    ax[1].set(xlabel="시간 단계", ylabel="채택 입력 열의 비용")
    finish_figure("suboptimal_mpc")
    print("실행 가능한 개선 수락 횟수:", accepted)


demo_suboptimal()


### 2.7.1 Extended State — 확장 상태

**교재 인쇄 p.150**

차선 MPC는 현재 공정 상태뿐 아니라 이전에 구한 실행 가능한 입력 열도 다음 결정에 사용합니다. 따라서 분석할 때에는 상태와 저장된 입력 열을 함께 확장 상태로 취급합니다. 같은 공정 상태라도 저장된 입력 열에 따라 다른 입력을 선택할 수 있습니다.

### 2.7.2 Asymptotic Stability of Difference Inclusions — 차분 포함 관계의 점근 안정성

**교재 인쇄 p.150–153**

조건을 만족하는 차선 입력 열은 하나가 아닐 수 있습니다. 가능한 다음 확장 상태가 여러 개라면 단일 함수 대신 집합값 관계로 표현합니다. 안정성 조건은 특정 솔버가 고른 한 경로뿐 아니라 허용되는 모든 선택에 대해 확인해야 합니다.

## 2.8 Economic Model Predictive Control — 경제적 모델 예측 제어

**교재 인쇄 p.153–155**

경제적 MPC는 목표 상태와의 거리를 벌점화하는 비용 대신 생산·에너지·운영 비용 자체를 예측 구간에서 최소화합니다. 최적 정상상태 $(x_s,u_s)$는 정상상태 방정식을 만족하는 상태·입력 중 경제적 비용이 가장 낮은 점입니다.

경제적 단계 비용은 정상상태에 대해 양의 정부호일 필요가 없으며 음수일 수도 있습니다. 비용을 최소화한다는 사실만으로 상태 수렴이 보장되지는 않습니다.

$$
(x_s,u_s)\in\arg\min_{(x,u)\in Z}\{\ell(x,u)\mid x=f(x,u)\}
$$

### 2.8.1 Asymptotic Average Performance — 점근 평균 성능

**교재 인쇄 p.155–156**

종단 상태를 고정된 정상상태로 제한하는 간단한 구성에서는 최적 비용의 차이를 시간에 걸쳐 더해 평균 성능 경계를 얻습니다. 이때 필요한 연속성·실행 가능성·비용 하한 등의 조건을 함께 사용합니다.

이 결과는 평균 경제적 비용에 관한 것이며 상태가 정상상태로 수렴한다는 뜻은 아닙니다. 유한 길이의 실험 평균에는 초기 과도응답이 포함되므로 점근 경계와 구분해야 합니다.

### 2.8.2 Dissipativity and Asymptotic Stability — 소산성과 점근 안정성

**교재 인쇄 p.156–160**

저장 함수 $\lambda$의 변화를 경제적 비용에서 보정하면 변환된 단계 비용 $L$을 만들 수 있습니다. 엄격한 소산성은 이 변환 비용이 정상상태로부터의 편차를 아래에서 제한하게 합니다. 적절한 연속성 등 추가 조건과 결합하여 안정성을 보입니다.

종단 상태가 고정되어 있으면 원래 구간 비용과 변환 비용은 입력 선택과 무관한 상수만큼 차이 납니다. 따라서 최적 입력은 같지만 안정성 분석에는 변환된 값 함수가 더 적합합니다.

$$
L(x,u)=\ell(x,u)-\ell(x_s,u_s)+\lambda(x)-\lambda(f(x,u))
$$

**보충 실습 E — 계산으로 확인하는 소산성**

$x^+=0.8x+0.2u$, $0\le x\le5$, $0\le u\le4$에서
$\ell(x,u)=0.5x^2+0.2u^2-2.8x$를 최소화합니다.
정상상태는 $x=u$이므로 경제적 최적점은 $(2,2)$이며 $\ell_s=-2.8$입니다.
저장 함수 $\lambda(x)=4x$를 사용하면 직접 전개하여

$$L(x,u)=0.5(x-2)^2+0.2(u-2)^2$$

를 얻습니다. 이 보충 예제에서 변환 비용이 이차 추종 비용과 같다는 것은 **특수한 구성의 결과**이며 일반적인 경제적 MPC의 성능 우위를 보여 주는 비교는 아닙니다.

In [ ]:
def demo_economic():
    N = 10
    xs = us = 2.0
    ell_s = -2.8
    x = cp.Variable(N + 1)
    u = cp.Variable(N)
    init = cp.Parameter()
    cons = [
        x[0] == init,
        x[1:] == 0.8 * x[:-1] + 0.2 * u,
        x >= 0,
        x <= 5,
        u >= 0,
        u <= 4,
        x[-1] == xs,
    ]
    economic = 0.5 * cp.sum_squares(x[:-1]) + 0.2 * cp.sum_squares(u) - 2.8 * cp.sum(x[:-1])
    rotated = 0.5 * cp.sum_squares(x[:-1] - xs) + 0.2 * cp.sum_squares(u - us)
    pe = cp.Problem(cp.Minimize(economic), cons)
    pr = cp.Problem(cp.Minimize(rotated), cons)

    def solve(x0, problem):
        init.value = float(x0)
        checked_solve(problem)
        assert abs(x.value[-1] - xs) < 1e-6
        assert min(x.value.min(), u.value.min()) > -1e-6
        assert x.value.max() < 5 + 1e-6 and u.value.max() < 4 + 1e-6
        return u.value.copy(), float(problem.value)

    differences = []
    for x0 in [0.2, 1.0, 2.0, 4.0]:
        ue, ve = solve(x0, pe)
        ur, vr = solve(x0, pr)
        differences.append(np.max(np.abs(ue - ur)))
        assert abs(vr - (ve - N * ell_s + 4 * x0 - 4 * xs)) < 1e-5
    assert max(differences) < 1e-5
    current = 0.2
    states = [current]
    costs = []
    values = []
    rv = []
    for k in range(70):
        seq, value = solve(current, pe)
        uk = seq[0]
        values.append(value)
        rv.append(value - N * ell_s + 4 * current - 4 * xs)
        costs.append(0.5 * current**2 + 0.2 * uk**2 - 2.8 * current)
        current = 0.8 * current + 0.2 * uk
        states.append(current)
    _, last = solve(current, pe)
    mean = np.mean(costs)
    bound = ell_s + (values[0] - last) / len(costs)
    assert mean <= bound + 1e-6
    assert max(np.diff(rv)) < 2e-6 and abs(states[-1] - xs) < 1e-6
    rng = np.random.default_rng(7)
    xx = rng.uniform(0, 5, 100)
    uu = rng.uniform(0, 4, 100)
    lhs = 0.5 * xx**2 + 0.2 * uu**2 - 2.8 * xx - ell_s + 4 * xx - 4 * (0.8 * xx + 0.2 * uu)
    rhs = 0.5 * (xx - xs) ** 2 + 0.2 * (uu - us) ** 2
    assert np.max(np.abs(lhs - rhs)) < 1e-12
    CHECKS["경제적·변환 비용 최적 입력 차이"] = float(max(differences))
    CHECKS["소산성 항등식 잔차"] = float(np.max(np.abs(lhs - rhs)))
    CHECKS["평균 성능 부등식 잔차"] = float(mean - bound)
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
    ax[0].plot(states)
    ax[0].axhline(xs, color="gray", ls=":")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[1].plot(np.cumsum(costs) / np.arange(1, len(costs) + 1), label="유한 시간 평균")
    ax[1].axhline(ell_s, color="gray", ls=":", label="최적 정상상태 비용")
    ax[1].set(xlabel="시간 단계", ylabel="평균 경제적 비용")
    ax[1].legend(fontsize=8)
    ax[2].semilogy(np.maximum(rv, 1e-12))
    ax[2].set(xlabel="시간 단계", ylabel="변환된 최적 값 함수")
    finish_figure("economic_mpc")
    print("실험 평균 비용:", mean, "유한 시간 상한:", bound)


demo_economic()


**보충 실습 — 종단 상태를 고정하지 않으면 비용 변환이 달라진다 (pp.156–160)**

합을 전개하면 $\sum L=\sum\ell-N\ell_s+\lambda(x_0)-\lambda(x_N)$입니다. 고정 종단에서는 마지막 항도 상수지만 자유 종단에서는 입력에 따라 달라집니다. 따라서 변환 비용만 최소화하면 원래 문제와 다른 입력을 얻을 수 있습니다. 같은 문제를 유지하려면 변환 비용에 종단 보정 $+\lambda(x_N)$을 추가해야 합니다. 아래는 동일 모델·동일 제약·$N=3$, $x_0=0.2$로 세 목적함수를 비교합니다.

In [ ]:
def demo_economic_terminal_term():
    N = 3
    x = cp.Variable(N + 1)
    u = cp.Variable(N)
    x0 = 0.2
    cons = [x[0] == x0, x[1:] == 0.8 * x[:-1] + 0.2 * u, x >= 0, x <= 5, u >= 0, u <= 4]
    economic = 0.5 * cp.sum_squares(x[:-1]) + 0.2 * cp.sum_squares(u) - 2.8 * cp.sum(x[:-1])
    rotated = 0.5 * cp.sum_squares(x[:-1] - 2) + 0.2 * cp.sum_squares(u - 2)
    results = []
    for label, cost in [
        ("경제적 비용", economic),
        ("변환 비용만", rotated),
        ("변환 + 종단 보정", rotated + 4 * x[-1]),
    ]:
        problem = cp.Problem(cp.Minimize(cost), cons)
        checked_solve(problem)
        results.append((label, x.value.copy(), u.value.copy()))
    restored = np.max(np.abs(results[0][2] - results[2][2]))
    changed = np.max(np.abs(results[0][2] - results[1][2]))
    assert restored < 1e-6 and changed > 0.1
    CHECKS["자유 종단 변환 비용 입력 차이"] = float(changed)
    CHECKS["종단 보정 후 입력 일치 오차"] = float(restored)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for i, (label, xs, us) in enumerate(results):
        ax[0].plot(xs, ["-", "--", ":"][i], marker="o", label=label)
        ax[1].plot(us, ["-", "--", ":"][i], marker="o", label=label)
    ax[0].set(xlabel="예측 단계", ylabel="상태")
    ax[1].set(xlabel="예측 단계", ylabel="입력")
    for axis in ax:
        axis.legend(fontsize=9)
    finish_figure("economic_terminal_term")
    return results


economic_terminal_result = demo_economic_terminal_term()


## 2.9 Discrete Actuators — 이산 구동기

**교재 인쇄 p.160–163**

켜짐·꺼짐이나 장비 대수처럼 입력이 이산값을 가지면 허용 집합은 비볼록일 수 있습니다. 기존 안정성 논리 중 집합의 닫힘·불변성 등에 근거한 부분은 필요한 가정을 충족하면 유지되지만, 계산은 더 어려워질 수 있습니다. 연속 완화 해를 단순 반올림하면 제약과 최적성을 잃을 수 있습니다.

**보충 실습 F — 작은 이산 입력 문제의 전수 탐색**

모델은 $x^+=0.8x+u$, 입력은 $u\in\{-1,0,1\}$, 상태 한계는 $|x|\le4$입니다.
종단 집합 $|x_N|\le0.5$에서 종단 제어 $u=0$이 불변성을 보장하고,
$P=1/(1-0.8^2)$는 그때의 미래 상태 비용입니다. 혼합정수 솔버 대신 짧은 입력 열을 전수 탐색합니다.

In [ ]:
def demo_discrete():
    N = 5
    a = 0.8
    r = 0.1
    P = 1 / (1 - a * a)
    sequences = np.array(list(product([-1.0, 0.0, 1.0], repeat=N)))
    relaxed = make_scalar_mpc(N=N, a=a, r=r, P=P, xf=0.5)
    x = 2.8
    states = [x]
    inputs = []
    values = []
    gaps = []
    for k in range(25):
        X = np.empty((len(sequences), N + 1))
        X[:, 0] = x
        for j in range(N):
            X[:, j + 1] = a * X[:, j] + sequences[:, j]
        valid = (np.max(np.abs(X), axis=1) <= 4 + 1e-10) & (np.abs(X[:, -1]) <= 0.5 + 1e-10)
        assert np.any(valid)
        J = (
            np.sum(X[:, :-1] ** 2, axis=1)
            + r * np.sum(sequences**2, axis=1)
            + P * X[:, -1] ** 2
        )
        idx = np.argmin(np.where(valid, J, np.inf))
        v = J[idx]
        uk = sequences[idx, 0]
        _, _, lower = relaxed(x)
        assert lower <= v + 1e-6
        gaps.append(v - lower)
        if values:
            assert v - values[-1] + states[-2] ** 2 + r * inputs[-1] ** 2 < 1e-6
        values.append(v)
        inputs.append(uk)
        x = a * x + uk
        states.append(x)
    assert set(inputs) <= set([-1.0, 0.0, 1.0]) and abs(states[-1]) < 0.01
    CHECKS["이산 입력 마지막 상태"] = abs(float(states[-1]))
    CHECKS["이산·연속 비용 격차 최솟값"] = float(min(gaps))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(states, "o-")
    ax[0].axhspan(-0.5, 0.5, alpha=0.12, color="#168b86", label="종단 집합")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[0].legend()
    ax[1].step(range(len(inputs)), inputs, where="post")
    ax[1].set(xlabel="시간 단계", ylabel="이산 입력", yticks=[-1, 0, 1])
    finish_figure("discrete_actuators")


demo_discrete()


**보충 실습 — 반올림은 종단 제약을 지키지 못할 수 있다 (pp.160–163 연결)**

같은 이산 입력 모델에서 $N=1$, $x_0=0.64$를 선택합니다. 연속 완화 최적 입력은 약 -0.494이므로 최근접 정수 0으로 반올림됩니다. 이때 다음 상태는 0.512로 종단 상한 0.5를 넘지만, 이산 최적 입력 -1은 다음 상태 -0.488로 실행 가능합니다. 완화 비용은 이산 최적 비용의 하한이며, 반올림 결과는 실행 가능성을 따로 검사해야 합니다.

In [ ]:
def demo_rounding_failure():
    a = 0.8
    x0 = 0.64
    r = 0.1
    P = 1 / (1 - a * a)
    solve = make_scalar_mpc(N=1, a=a, r=r, P=P, xf=0.5)
    xp, continuous, lower = solve(x0)
    rounded = np.rint(continuous)
    options = np.array([-1.0, 0.0, 1.0])
    states = a * x0 + options
    valid = np.abs(states) <= 0.5 + 1e-12
    cost = x0 * x0 + r * options**2 + P * states**2
    idx = np.argmin(np.where(valid, cost, np.inf))
    exact = options[idx]
    rounded_state = float(a * x0 + rounded[0])
    violation = abs(rounded_state) - 0.5
    assert violation > 0.01 and exact == -1 and lower <= cost[idx] + 1e-7
    CHECKS["반올림 입력 종단 위반량"] = float(violation)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    labels = ["연속 완화", "반올림", "이산 최적"]
    ax[0].bar(
        labels, [continuous[0], rounded[0], exact], color=["#168b86", "#dd6644", "#13293d"]
    )
    ax[1].bar(
        labels,
        [xp[-1], rounded_state, states[idx]],
        color=["#168b86", "#dd6644", "#13293d"],
    )
    ax[0].set(ylabel="입력")
    ax[1].set(ylabel="다음 상태")
    ax[1].axhspan(-0.5, 0.5, alpha=0.1, color="green")
    ax[1].axhline(0.5, color="gray", ls="--")
    ax[1].axhline(-0.5, color="gray", ls="--")
    finish_figure("rounding_failure")
    return {
        "continuous": continuous[0],
        "rounded": rounded[0],
        "exact": exact,
        "violation": violation,
    }


rounding_result = demo_rounding_failure()


## 2.10 Concluding Comments — 정리

**교재 인쇄 p.163–166**

MPC 설계는 모델과 비용의 선택만으로 끝나지 않습니다. 실행 가능성, 종단 설계, 값 함수의 경계와 감소, 실제 솔버가 반환하는 해의 성질을 함께 확인해야 합니다. 불확실성이 있는 실제 시스템으로 옮길 때에는 명목 조건에서 얻은 결론의 적용 범위를 다시 검토해야 합니다.

## 2.11 Notes — 문헌 해설

**교재 인쇄 p.166–172**

이 절은 교재가 관련 연구와 확장 결과를 정리하는 문헌 해설입니다. 여기서는 새로운 알고리즘을 추가하기보다 각 결과를 읽을 때 확인할 기준을 제시합니다. 특히 명목 안정성과 강인 안정성, 종단 제약 유무, 전역 최적해 요구 여부를 구분해서 읽어야 합니다.

## 2.12 Exercises — 연습문제

**교재 인쇄 p.172–192**

교재 연습문제는 원래 번호로 참조합니다. 연습문제 2.3(p.172)은 상태·출력 제약이 안정성에 미치는 영향, 2.4(pp.172–173)는 실행 가능 집합의 투영, 2.5(p.173)는 최대 출력 허용 집합을 다룹니다. 아래에는 이번 노트북 실습을 확장하는 보충 과제를 제공합니다.

### 보충 과제 A — 종단 집합 크기
`demo_terminal_radius()`의 최대 반경을 손으로 계산하고 경계점 검사와 대조하세요. `demo_terminal_radius(enlargement=1.2)`를 실행하기 전에 이동 입력 위반량이 얼마가 될지 예측하세요. 비용 감소 항등식이 성립하는데도 이 고정 종단 제어의 안정성 증명을 그대로 쓸 수 없는 이유를 설명하세요.

### 보충 과제 B — 외란으로 이동 후보가 깨지는 경우
실제 다음 상태에만 일정한 외란을 더하세요. 명목 예측을 이동한 후보가 왜 그대로 실행 가능하다고 할 수 없는지 확인하세요.
이 실험의 실패를 3장에서 다룰 문제와 연결하세요.

### 보충 과제 C — 주기 위상
주기 모델 실습의 종단 행렬을 종료 위상이 아니라 현재 위상의 것으로 바꾸세요.
QP와 주기 리카티 입력이 언제 달라지는지 확인하세요.

### 보충 과제 D — 차선해 수락 규칙
실행 가능성 검사를 제거하거나 비용이 증가하는 변경도 채택하면 무엇을 잃는지 설명하세요.
안정성 검증에서 공정 상태와 저장된 입력 열을 함께 고려하는 이유를 정리하세요.

### 보충 과제 E — 경제적 비용 변환
$\lambda(x)=4x$를 대입해 변환 비용을 손으로 전개하세요.
종단 등식을 제거하면 변환 전후 비용 차이가 여전히 입력 열과 무관한 상수인지 확인하세요.

### 보충 과제 F — 이산 입력 계산량
예측 길이를 5에서 8로 바꾸면 후보가 몇 배 늘어나는지 계산하세요.
연속 완화 입력을 반올림했을 때 종단 제약을 만족하는지도 검사하세요.

<details><summary>힌트</summary>

- A: 스칼라 종단 피드백의 입력 조건은 반경이 $1/|K|$ 이하일 때 성립합니다. 불변성도 따로 확인합니다.
- B: 이동 논리는 실제 다음 상태가 예측 첫 상태와 일치할 때 그대로 적용됩니다.
- C: 비용의 마지막 항은 예측 종료 시점의 모델 위상과 연결됩니다.
- D: 같은 공정 상태라도 저장된 후보가 달라지면 다음 제어 입력이 달라질 수 있습니다.
- E: 종단 상태가 자유로우면 $-\lambda(x_N)$가 입력 선택에 의존합니다.
- F: $3^8/3^5=27$배입니다. 반올림은 동역학을 통해 모든 예측 상태를 바꿉니다.

</details>

## 실행 검증 결과

각 실습은 수치 허용오차가 있는 `assert`로 기본 설정을 검증합니다. 다음 값은 이 실행에서 측정된 지표입니다.
단일 실험의 성공은 모든 모델·초기상태에서의 안정성을 증명하지 않습니다.

In [ ]:
import json

print(json.dumps(CHECKS, ensure_ascii=False, indent=2))
print(f"기록된 검증 지표: {len(CHECKS)}개")


## 참고 자료와 표기

- Rawlings, Mayne, Diehl (2017), *Model Predictive Control: Theory, Computation, and Design*, 2nd edition, 1st printing, Chapter 2, pp.89–192.
- 설명은 한국어로 재서술했으며 PPT·노트북 모두 교재 절 순서를 따릅니다.
- Examples 2.5–2.6은 원본 수치 사례를 재현하고, Example 2.8은 실행 가능 구조를 계산했습니다. 나머지는 보충 실습입니다. 교재의 모든 수치 사례·증명·연습문제를 재현한 자료는 아닙니다.
- 제어 이득은 `u=Kx` 관례입니다. Examples 2.5–2.6은 원본의 1/2를 유지합니다. 나머지 보충 LQ 실습은 공통 계수 1/2를 생략했으며, 값 함수 감소 검사에도 같은 비용 정의를 사용합니다.
- 한글 글꼴: Google Fonts의 나눔고딕. 설치 및 첫 글꼴 다운로드에는 인터넷 연결이 필요합니다.